# AAVAIL revenue EDA
Run `python ingest.py` after putting the course data in `data/cs-train`.

In [ ]:
import sys; sys.path.insert(0, '..')
import matplotlib.pyplot as plt
from ingest import fetch_data, fetch_ts, TRAIN_DIR
df = fetch_data(TRAIN_DIR)
df.info(); df.describe()

In [ ]:
print('missing values:'); print(df.isna().sum())
print('date range:', df.invoice_date.min(), '->', df.invoice_date.max())

In [ ]:
top = df.groupby('country').price.sum().nlargest(10)
top.plot.barh(title='Top 10 countries by revenue'); plt.gca().invert_yaxis(); plt.show()
(top / df.price.sum()).round(3)

In [ ]:
ts = fetch_ts(TRAIN_DIR)
fig, ax = plt.subplots(figsize=(12, 4))
ts['all'].set_index('date').revenue.rolling(7).mean().plot(ax=ax, title='Daily revenue (7-day mean), all countries')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for k in list(ts)[:6]:
    if k != 'all': ts[k].set_index('date').revenue.resample('M').sum().plot(ax=ax, label=k)
ax.legend(); ax.set_title('Monthly revenue by top country'); plt.show()

In [ ]:
a = ts['all'].set_index('date')
a.groupby(a.index.dayofweek).revenue.mean().plot.bar(title='Mean revenue by weekday (0=Mon)'); plt.show()
a[['purchases','unique_invoices','total_views','revenue']].corr()

## Findings
- Revenue is dominated by the United Kingdom; the top 10 countries cover almost all revenue.
- Strong weekly pattern and seasonal peaks late in the year, so lag/window features (7, 14, 28, 70 days, last year) are used for modeling.
- Views and invoices correlate strongly with revenue.